In [3]:
import numpy as np
import pandas as pd
import spacy
from sklearn.linear_model import LogisticRegression

# Install the spaCy model if it's not found
!python -m spacy download en_core_web_lg

# Загрузка модели spaCy для токенизации
nlp = spacy.load("en_core_web_lg")

# Пример предобработки текста
def preprocess_text(text):
    # Преобразуем текст в нижний регистр и токенизируем с удалением пунктуации и стоп-слов
    doc = nlp(text.lower())
    tokens = [token.lemma_ for token in doc if not token.is_stop and not token.is_punct]
    return tokens

# Функция для получения вектора отзыва через mean pooling
def get_mean_vector(tokens):
    vectors = [nlp(word).vector for word in tokens]
    return np.mean(vectors, axis=0) if vectors else np.zeros((nlp.vocab.vectors.shape[1],))

# Обучение классификатора (например, логистическая регрессия)
# Предположим, что clf уже обучен на векторах
# Пример: обучаем классификатор на случайных векторах и метках
# Здесь предполагается, что у вас уже есть обученная модель clf

# Данные для обучения классификатора (пример)
# В реальности нужно будет обучить модель на данных IMDB
train_data = [
    ("The movie was good.", "positive"),
    ("The movie was bad.", "negative"),
    ("I loved this film.", "positive"),
    ("I hated this film.", "negative")
]

# Преобразуем данные в векторы
X_train = np.array([get_mean_vector(preprocess_text(text)) for text, _ in train_data])
y_train = ["positive" if label == "positive" else "negative" for _, label in train_data]

# Обучаем классификатор
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

# Функция для предсказания
def predict_sentence(sentence):
    tokens = preprocess_text(sentence)
    vec = get_mean_vector(tokens).reshape(1, -1)
    prob_pos = clf.predict_proba(vec)[0, 1]
    pred = "positive" if prob_pos >= 0.5 else "negative"
    return tokens, pred, prob_pos

# Ваши контрастные примеры
contrast_sentences = [
    # negation
    "The movie was good.",
    "The movie was not good.",
    "I did not enjoy this film.",
    "I never hated the acting.",
    # word order
    "Dog bites man.",
    "Man bites dog.",
    "Flights from Astana to Almaty are cheap.",
    "Flights from Almaty to Astana are cheap.",
    # ambiguity / word sense
    "He went to the bank to deposit money.",
    "He sat by the river bank.",
    # translation / bilingual inspired
    "Translate this book.",
    "Book this flight.",
    # own choice
    "This movie is so bad that it is good.",
    "I hardly liked it."
]

# Применяем классификатор к контрастным примерам
rows = []
for sent in contrast_sentences:
    tokens, pred, prob_pos = predict_sentence(sent)
    rows.append({
        "sentence": sent,
        "tokens": tokens,
        "predicted_label": pred,
        "positive_probability": round(float(prob_pos), 4)
    })

# Создаем DataFrame для удобства отображения
contrast_df = pd.DataFrame(rows)
print(contrast_df)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.7/400.7 MB 4.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
                                    sentence                           tokens  \
0                        The movie was good.                    [movie, good]   
1                    The movie was not good.                    [movie, good]   
2                 I did not enjoy this film.                    [enjoy, film]   
3                  I never hated the acting.                    [hat, acting]   
4                             Dog bites man.                 [dog, bite, man]   
5                             Man bites dog.                 [man, bite, dog]   
6   Flights from

In [4]:

print("PART D2: 4 Important Failure Cases\n")

print("1. Negation failure")
print("   Example: 'The movie was good.' vs 'The movie was not good.'")
print("   Problem: the word 'good' can dominate the sentence vector, while negation is only weakly represented.")
print()

print("2. Word-order failure")
print("   Example: 'Dog bites man.' vs 'Man bites dog.'")
print("   Problem: both sentences contain the same words, so their average embedding is the same.")
print()

print("3. Polysemy / ambiguity")
print("   Example: 'He went to the bank to deposit money.' vs 'He sat by the river bank.'")
print("   Problem: the word 'bank' has one static vector, even though its meaning changes with context.")
print()

print("4. Command / noun ambiguity")
print("   Example: 'Translate this book.' vs 'Book this flight.'")
print("   Problem: the word 'book' can be a noun or a verb, but a static embedding does not fully resolve that difference.")
print()


PART D2: 4 Important Failure Cases

1. Negation failure
   Example: 'The movie was good.' vs 'The movie was not good.'
   Problem: the word 'good' can dominate the sentence vector, while negation is only weakly represented.

2. Word-order failure
   Example: 'Dog bites man.' vs 'Man bites dog.'
   Problem: both sentences contain the same words, so their average embedding is the same.

3. Polysemy / ambiguity
   Example: 'He went to the bank to deposit money.' vs 'He sat by the river bank.'
   Problem: the word 'bank' has one static vector, even though its meaning changes with context.

4. Command / noun ambiguity
   Example: 'Translate this book.' vs 'Book this flight.'
   Problem: the word 'book' can be a noun or a verb, but a static embedding does not fully resolve that difference.

